# Cross-modal geometry: full-data description, then held-out trials
Run `compare_subspace.py` first. Conditions are equally averaged after trial averaging. The all-data fit is explicitly in-sample (repeat -1); it is followed by five folds with disjoint test trials, fixed cohort and matching, and overlapping training sets. There is no tuning or test A/B.

Temporal rows are time points. Spatial rows are electrode/source correspondences. Generalization concerns new trials at the same participants and locations.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'compare_subspace.py').exists() and (ROOT / 'iEEGvsMEG' / 'compare_subspace.py').exists():
    ROOT = ROOT / 'iEEGvsMEG'
sys.path.insert(0, str(ROOT))
from compare_subspace import load_results, plot_results

MEG_KIND = 'paired_coverage'  # full_average, full_concatenated, coverage_average, paired_coverage, random_control
OUTPUT_DIR = ROOT / 'out' / 'compare_subspace' / MEG_KIND
DISPLAY_K = 3 

## Run the analysis
```bash
python -u compare_subspace.py --root /path/to/iEEGvsMEG \
  --meg-kind paired_coverage --models separate_pca plssvd joint_pca \
  --dimensions 1 2 3 5 10 --n-splits 5 --block-scaling equal_variance \
  --ridge-alpha 0.01 --output-dir /path/to/new_geometry_results
```
Use a new directory if results already exist. Whole-modality scaling remains configurable; ridge is fixed, and all requested cluster counts are reported.


In [ ]:
config, tables = load_results(OUTPUT_DIR)
if config.get('schema_version', 1) < 2:
    raise ValueError('Run the updated analysis into a new directory: these outputs use the old split design.')
display(pd.Series(config, name='Run configuration'))
# Full-data fit first: descriptive scores, not testing or an upper bound.
for name in ('overlap', 'alignment', 'clusters'):
    print(f'All-data description: {name}')
    display(tables[name].query("partition == 'in_sample' and k == @DISPLAY_K"))
plot_results(OUTPUT_DIR, k=DISPLAY_K, show=False)


## 1. Subspace overlap
Principal-angle overlap compares the full retained spaces, permitting axis rotations/mixing. One means identical full-rank spaces and zero means orthogonal. Missing dimensions count as zero. Temporal and spatial similarity answer separate questions.

Dashed lines: all-data in-sample description. Solid lines: test-fold median. Shading: fold minimum–maximum, not confidence intervals.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'subspace_overlap.png')))
display(tables['overlap'].query("partition == 'test'")[
    ['model', 'repeat', 'space', 'k', 'overlap', 'rank_a', 'rank_b', 'angles_deg']])


## 2. Transformation generalization
Training means and one scalar RMS normalize each modality's representation. Fit identity, orthogonal, affine and quadratic maps in both directions. Ridge is fixed beforehand (default 0.01); no tuning. Apply each map unchanged to held-out responses.

NRMSE below 1 beats the training-mean baseline; Q² = 1 − NRMSE². Dashed all-data results are descriptive; use test-fold errors to assess generalization. Time points and locations recur across partitions.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / f'alignment_complexity_k{DISPLAY_K}.png')))
display(tables['alignment'].query("partition == 'test' and k == @DISPLAY_K")[
    ['model', 'repeat', 'space', 'direction', 'complexity', 'alpha', 'nrmse', 'q2']])


## 3. Fold consistency
Test folds do not overlap. Training folds do overlap, so the range across test scores is descriptive, not independent replication. There are no A/B reliability outputs.


In [ ]:
display(tables['alignment'].query("partition == 'test' and k == @DISPLAY_K").groupby(
    ['model', 'space', 'direction', 'complexity'])[['nrmse', 'q2']].agg(['median', 'min', 'max']))


## 4. Exploratory spatial grouping
K-means is fitted separately to each modality's training spatial patterns. All predeclared cluster counts are reported; none is selected using test agreement. Held-out patterns are assigned to training centroids. ARI compares cross-modal membership and silhouette describes compactness/separation. No test-half stability is computed. Stable groups do not by themselves establish biological networks.


In [ ]:
display(Image(filename=str(OUTPUT_DIR / f'cluster_agreement_k{DISPLAY_K}.png')))
display(tables['clusters'].query("partition == 'test' and k == @DISPLAY_K"))


## 5. Within-modality comparison (section 3.2)
Compare models within each modality on condition-averaged time courses and native-feature forward patterns. Component pairs/signs come from training time courses and remain fixed on testing. All-data rows are descriptive. Detailed pairs and correlations are exported separately.


In [ ]:
tables

In [ ]:
import matplotlib.pyplot as plt
from compare_models import plot_within_models

display(tables['within_model_metrics'].query("partition == 'test' and k == @DISPLAY_K"))
display(tables['within_model_pairs'].query('repeat == 0 and k == @DISPLAY_K'))
for index, fig in enumerate(plot_within_models(tables, k=DISPLAY_K, partition='test')):
    fig.savefig(OUTPUT_DIR / f'within_models_k{DISPLAY_K}_{index:02d}.png', dpi=160)
    display(fig)
    plt.close(fig)
